# SwiGLU MLP

源码导航：[`SwiGLUMLP`](../../../core/ffn/swiglu.py#L19)。

SwiGLU 把传统 FFN 的单路非线性改成“门控分支 × 内容分支”。给定输入 $x\in\mathbb{R}^{B\times T\times d}$：

$$
\operatorname{SwiGLU}(x)=W_{down}\left(\operatorname{SiLU}(W_{gate}x)\odot W_{up}x\right)
$$

它的改进点在于：门控分支可以动态调节通道信息流，通常在相近参数量下比 GELU MLP 表现更好。当前实现显式传入 `d_ffn`，便于在 1B 参数预算内控制 FFN 占比。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.ffn.swiglu import SwiGLUMLP

## 1. 前向形状

In [ ]:
torch.manual_seed(0)
mlp = SwiGLUMLP(n_embd=128, d_ffn=256, dropout=0.0, bias=False)
x = torch.randn(2, 8, 128)
y = mlp(x)
print('x:', tuple(x.shape))
print('y:', tuple(y.shape))

## 2. 参数构成

In [ ]:
for name, p in mlp.named_parameters():
    print(f'{name:18s}', tuple(p.shape), p.numel())
print('total:', sum(p.numel() for p in mlp.parameters()))

---

## 延伸阅读与参考资料

### 核心论文
- **GLU Variants Improve Transformer**: Shazeer, 2020. [arXiv:2002.05202](https://arxiv.org/abs/2002.05202)
- **PaLM**: Chowdhery et al., 2022. [arXiv:2204.02311](https://arxiv.org/abs/2204.02311)
- **LLaMA**: Touvron et al., 2023. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)

### 工程实现
- **Hugging Face Transformers LlamaMLP**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)
- **PyTorch SiLU API**: [docs](https://pytorch.org/docs/stable/generated/torch.nn.SiLU.html)